# 05 · Chốt cấu hình trên L4 — Qwen2.5-VL-3B (W7)

**Vấn đề:** cấu hình baseline W5 dùng **fp16**, và W6 đã chứng minh fp16 gây tràn số (23/2.000 câu ra `!!!!`). fp32 thì sửa được lỗi nhưng huấn luyện quá chậm trên T4 (~14,8 giờ/epoch). Vì vậy W6 hoãn **các quyết định cấu hình** sang W7, để đo trên GPU L4 có bf16 thật.

**Notebook này trả lời 4 câu hỏi, rồi mới được chạy B1:**

| Phần | Câu hỏi | Cách trả lời |
|---|---|---|
| 2 | L4 có bf16 *thật* không? | Đọc compute capability (≥ sm_80 mới có bf16 phần cứng) |
| 3 | Thông số kiến trúc baseline lấy từ đâu? | Đọc `config.json` chính thức trên HF, ghi kèm revision |
| 4–7 | Dùng dtype nào cho B1/B2? | Probe **fp16 / fp32 / bf16** trên đúng 73 câu probe của W6, luật chọn định trước |
| 8–9 | `max_pixels` 512 hay 1.280 token? Train 1 epoch mất bao lâu trên L4? | Đo chi phí suy luận + smoke train 30 step ở hai độ phân giải |

**Nguyên tắc:** notebook **không sửa** `configs/qwen_lora.yaml`. Nó chỉ sinh số liệu và một file tóm tắt. Cấu hình chuẩn được đổi sau, kèm lý do ghi vào R4.

**Điều kiện:** Runtime → Change runtime type → **L4 GPU** → Save. Mọi run từ W7 trở đi dùng **cùng loại GPU này**, để thời gian suy luận và huấn luyện so sánh được với nhau.

**Thời gian ước tính:** khoảng 1–1,5 giờ (chưa đo trên L4). Giữ tab mở. Nếu Colab rớt, chạy lại từ phần 1: các lô đã xong tự được bỏ qua.

## 1. Thiết lập môi trường

Mount Drive, clone/pull code mới nhất, cài thư viện (pin `transformers==5.17.0` giống W5/W6 để chỉ GPU và dtype thay đổi, thư viện thì không), nạp `HF_TOKEN` nếu có.

*Kỳ vọng: in commit mới nhất, GPU `NVIDIA L4`, `✅ Thiết lập xong`. Khoảng 1–2 phút.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU -> đổi runtime sang L4')
print('✅ Thiết lập xong')

## 2. Bằng chứng phần cứng: L4 có bf16 thật

**Vì sao không dùng `torch.cuda.is_bf16_supported()`:** ở W6, hàm này trả về `True` ngay cả trên T4. Lý do là PyTorch giả lập bf16 bằng phần mềm (rất chậm). Tiêu chí đúng là **compute capability ≥ 8.0** (Ampere trở lên), vì đó mới là bf16 chạy trên phần cứng. L4 là kiến trúc Ada, sm_89.

Cell này cũng tạo thư mục kết quả của notebook: `experiments/W07_config_freeze_L4_seed42/`.

*Kỳ vọng: `gpu: NVIDIA L4`, `compute_capability: sm_89`, `bf16_native: true`, VRAM khoảng 22 GB.*

In [ ]:
import json, platform

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
EXP_ROOT   = f'{DRIVE_ROOT}/experiments'
OUT_DIR    = f'{EXP_ROOT}/W07_config_freeze_L4_seed42'
os.makedirs(OUT_DIR, exist_ok=True)

gpu_name = torch.cuda.get_device_name(0)
cap      = torch.cuda.get_device_capability(0)
HW = {
    'gpu': gpu_name,
    'compute_capability': f'sm_{cap[0]}{cap[1]}',
    'bf16_native': cap[0] >= 8,                          # tiêu chí đúng (phần cứng)
    'bf16_supported_torch': bool(torch.cuda.is_bf16_supported()),  # chỉ để đối chiếu với W6
    'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
    'torch': torch.__version__,
    'platform': platform.platform(),
}
print(json.dumps(HW, indent=2, ensure_ascii=False))
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

if 'L4' not in gpu_name:
    print(f'\n⚠️ Đang chạy trên {gpu_name}, KHÔNG phải L4. Kết quả vẫn hợp lệ, nhưng mọi run W7+ phải dùng cùng loại GPU.')
if not HW['bf16_native']:
    print('⚠️ GPU này không có bf16 phần cứng -> nhánh bf16 ở phần 6 sẽ chạy giả lập, chậm và không đại diện.')

## 3. Bằng chứng kiến trúc: `config.json` chính thức của Qwen2.5-VL-3B

**Vấn đề:** sơ đồ baseline của nhóm lấy thông số từ `config.json` (file mà `from_pretrained` thực sự nạp). Bảng 1 của paper (Bai et al., 2025, arXiv:2502.13923) lệch ở 3 chỗ. Hội đồng có thể hỏi "lấy số này từ đâu?", nên cần một bằng chứng **có revision** (mã commit của repo HF), để ai cũng tải lại được đúng file đó.

Cell này: (1) tải `config.json` theo revision hiện tại; (2) in các thông số dùng trong sơ đồ; (3) đối chiếu 3 chỗ lệch với paper; (4) kiểm tra công thức **784 pixel / token**, tức (patch 14 × merge 2)²; (5) ghi file bằng chứng.

*Kỳ vọng: vision depth 32, full-attention ở block 7/15/23/31 (cấu trúc "7 window + 1 full" lặp 4 lần), 3 dòng đánh dấu `LỆCH` so với paper, và 256 / 512 / 1.280 token cho min / max hiện tại / max ablation.*

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

REPO_ID  = 'Qwen/Qwen2.5-VL-3B-Instruct'
revision = HfApi().model_info(REPO_ID).sha                  # mã commit của repo HF tại thời điểm tải
cfg_path = hf_hub_download(REPO_ID, 'config.json', revision=revision)
raw      = json.load(open(cfg_path, encoding='utf-8'))

v = raw['vision_config']
t = raw.get('text_config', raw)            # một số bản lồng thông số LLM trong text_config
def llm(key):
    return t.get(key, raw.get(key))

ARCH = {
    'vision.depth':                 v.get('depth'),
    'vision.hidden_size':           v.get('hidden_size'),
    'vision.intermediate_size':     v.get('intermediate_size'),
    'vision.num_heads':             v.get('num_heads'),
    'vision.patch_size':            v.get('patch_size'),
    'vision.temporal_patch_size':   v.get('temporal_patch_size'),
    'vision.spatial_merge_size':    v.get('spatial_merge_size'),
    'vision.window_size':           v.get('window_size'),
    'vision.fullatt_block_indexes': v.get('fullatt_block_indexes'),
    'vision.out_hidden_size':       v.get('out_hidden_size'),
    'llm.hidden_size':              llm('hidden_size'),
    'llm.intermediate_size':        llm('intermediate_size'),
    'llm.num_hidden_layers':        llm('num_hidden_layers'),
    'llm.num_attention_heads':      llm('num_attention_heads'),
    'llm.num_key_value_heads':      llm('num_key_value_heads'),
    'llm.vocab_size':               llm('vocab_size'),
    'llm.max_position_embeddings':  llm('max_position_embeddings'),
}
print(f'{REPO_ID} @ revision {revision}\n')
for k, val in ARCH.items():
    print(f'  {k:30s} = {val}')

# 3 chỗ Bảng 1 paper (bản 3B) lệch với config.json — nhóm lấy config.json làm chuẩn
PAPER_TABLE1 = {'vision.intermediate_size': 3456, 'llm.intermediate_size': 4864, 'llm.vocab_size': 151646}
print('\nĐối chiếu với Bảng 1 paper:')
for k, paper_val in PAPER_TABLE1.items():
    flag = 'LỆCH' if ARCH[k] != paper_val else 'khớp'
    print(f'  {k:30s} config={ARCH[k]!s:>8}  paper={paper_val!s:>8}  -> {flag}')

# Công thức token ảnh: mỗi token = 1 khối (patch × merge)² pixel
px_per_token = (ARCH['vision.patch_size'] * ARCH['vision.spatial_merge_size']) ** 2
print(f'\nPixel / token ảnh = ({ARCH["vision.patch_size"]} × {ARCH["vision.spatial_merge_size"]})² = {px_per_token}')
for name, px in [('min_pixels', 200704), ('max_pixels W5', 401408), ('max_pixels ablation', 1003520)]:
    print(f'  {name:20s} {px:>9,} px -> {px // px_per_token:>5} token')
assert px_per_token == 784, 'Công thức token khác 784 -> kiểm tra lại sơ đồ'

with open(f'{OUT_DIR}/qwen25vl3b_config_evidence.json', 'w', encoding='utf-8') as fp:
    json.dump({'repo_id': REPO_ID, 'revision': revision, 'arch': ARCH,
               'paper_table1_diffs': PAPER_TABLE1, 'px_per_token': px_per_token,
               'config_json_raw': raw}, fp, ensure_ascii=False, indent=2)
print('\nGhi ->', f'{OUT_DIR}/qwen25vl3b_config_evidence.json')

## 4. Nạp tập probe 73 câu và kiểm tra đúng file của W6

**Vì sao phải kiểm tra:** để so sánh L4 với T4, cả hai phải chạy trên **cùng một file probe**. File này gồm 23 câu hỏng và 50 câu đối chứng, đã sinh ở notebook 03. Cell so `sha256` của file trên Drive với giá trị ghi trong log W6 (đã commit trong repo). Lệch là phải dừng.

*Kỳ vọng: `73 mẫu`, `23 hỏng + 50 đối chứng`, `✅ sha256 khớp W6`.*

In [ ]:
import hashlib

DATA_DIR   = f'{DRIVE_ROOT}/data/vitextvqa_official'
PROBE_PATH = f'{DATA_DIR}/probe_dtype73_seed42.json'
W6_LOG     = 'experiments/W06_probe_fp32_qwen25vl_vitextvqa73_seed42/metrics.json'   # nằm trong repo

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fp:
        for block in iter(lambda: fp.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

assert os.path.exists(PROBE_PATH), f'Thiếu {PROBE_PATH} -> chạy lại phần 3 của notebook 03'
probe      = json.load(open(PROBE_PATH, encoding='utf-8'))
failed     = json.load(open('experiments/W06_dtype_probe_seed42/failed_ids.json', encoding='utf-8'))
FAILED_IDS = set(map(str, failed['question_ids']))
PROBE_IDS  = [str(r['question_id']) for r in probe]
CONTROL_IDS = [q for q in PROBE_IDS if q not in FAILED_IDS]

print(f'{len(probe)} mẫu = {len(FAILED_IDS)} hỏng + {len(CONTROL_IDS)} đối chứng')
expected_sha = json.load(open(W6_LOG, encoding='utf-8'))['meta']['data_file_sha256']
actual_sha   = sha256_file(PROBE_PATH)
assert actual_sha == expected_sha, f'sha256 lệch W6: {actual_sha[:12]} vs {expected_sha[:12]} -> KHÔNG so sánh được'
print('✅ sha256 khớp W6:', actual_sha[:16], '…')

## 5. Hàm chạy một biến thể cấu hình

Giống notebook 03: mỗi biến thể **sinh một file config tạm** trong `/content/cfg_variants/` rồi gọi `run_baseline.run(..., check_config=False)`. Nhờ vậy mọi biến thể đi qua **cùng một đường code**, chỉ khác config.

Ba hàm:
- `make_cfg(...)` — sinh file config biến thể.
- `run_variant(...)` — chạy và bắt lỗi hết VRAM (OOM) thay vì để notebook chết giữa chừng.
- `em_by_id(...)` — điểm EM từng câu, dùng để đếm câu **thoái lui** giữa hai dtype.

*Kỳ vọng: chỉ in `✅ Hàm đã sẵn sàng.`, chưa dùng GPU.*

In [ ]:
import gc, time, yaml
from src.eval import run_baseline
from src.eval.find_degraded import is_degraded
from src.eval.metrics import exact_match

BASE_CFG = 'configs/qwen_lora.yaml'
CFG_DIR  = '/content/cfg_variants'
os.makedirs(CFG_DIR, exist_ok=True)
RESULTS  = []

def make_cfg(name, dtype, max_pixels=401408, load_in_4bit=True):
    cfg = yaml.safe_load(open(BASE_CFG, encoding='utf-8'))
    cfg['model']['torch_dtype'] = dtype                     # phần không lượng tử hoá (vision tower)
    cfg['model']['max_pixels']  = max_pixels
    cfg['quantization']['enabled']                = load_in_4bit
    cfg['quantization']['load_in_4bit']           = load_in_4bit
    cfg['quantization']['bnb_4bit_compute_dtype'] = dtype   # phần lượng tử hoá 4-bit
    path = f'{CFG_DIR}/{name}.yaml'
    with open(path, 'w', encoding='utf-8') as fp:
        yaml.safe_dump(cfg, fp, allow_unicode=True)
    return path

def load_preds(run_name):
    return json.load(open(f'{EXP_ROOT}/{run_name}/predictions.json', encoding='utf-8'))

def em_by_id(run_name):
    return {str(r['question_id']): exact_match(r['prediction'], r['answers']) for r in load_preds(run_name)}

def run_variant(run_name, dtype, max_pixels=401408, chunk=25):
    cfg_path = make_cfg(run_name, dtype, max_pixels)
    print(f'\n{"=" * 70}\n▶ {run_name} | dtype={dtype} | max_pixels={max_pixels}\n{"=" * 70}')
    t0 = time.time()
    try:
        res = run_baseline.run(dataset='vitextvqa_official', run_name=run_name,
                               drive_root=DRIVE_ROOT, data_file=PROBE_PATH, out_root=EXP_ROOT,
                               config=cfg_path, chunk=chunk, check_config=False)
        bad = [str(r['question_id']) for r in load_preds(run_name) if is_degraded(r['prediction'])]
        out = {'run_name': run_name, 'status': 'ok', 'dtype': dtype, 'max_pixels': max_pixels,
               'exact_match': round(res['metrics']['exact_match'], 4),
               'anls': round(res['metrics']['anls'], 4),
               'n_degraded': len(bad), 'degraded_ids': bad,
               'sec_per_sample': res['meta']['sec_per_sample'],
               'elapsed_sec': round(time.time() - t0, 1)}
    except torch.cuda.OutOfMemoryError as exc:
        out = {'run_name': run_name, 'status': 'OOM', 'dtype': dtype, 'max_pixels': max_pixels,
               'error': str(exc)[:200]}
        print('⚠️ HẾT VRAM — ghi nhận, chạy tiếp biến thể sau.')
    finally:
        gc.collect(); torch.cuda.empty_cache()
    RESULTS.append(out)
    print({k: v for k, v in out.items() if k != 'degraded_ids'})
    return out

print('✅ Hàm đã sẵn sàng.')

## 6. Probe dtype trên L4: fp16 / fp32 / bf16 (512 token)

Ba nhánh, chỉ khác **một thứ** là kiểu số khi tính toán. Weight vẫn nén 4-bit NF4 ở cả ba, nên ràng buộc PEFT/efficiency không đổi.

| Nhánh | Mục đích |
|---|---|
| fp16 | Lỗi tràn số có tái hiện trên L4 không? (W6 trên T4: 23 câu hỏng) |
| fp32 | Mốc an toàn, cũng là cấu hình W6 đã chọn |
| bf16 | Ứng viên: dải số rộng như fp32, tốc độ gần fp16 |

Mỗi nhánh nạp model một lần. Lần đầu mất vài phút vì phải tải weight, các lần sau nhanh hơn nhờ cache.

*Kỳ vọng: 3 dòng kết quả, mỗi nhánh vài phút. Giữ tab mở.*

In [ ]:
for dt, tag in [('float16', 'fp16'), ('float32', 'fp32'), ('bfloat16', 'bf16')]:
    run_variant(f'W07_probe_{tag}_qwen25vl_vitextvqa73_L4_seed42', dtype=dt)

## 7. Kết luận dtype — theo luật định trước

**Luật được viết TRƯỚC khi nhìn số** (tránh nhìn kết quả rồi mới nghĩ ra lý do):

1. Chọn **bf16** nếu: không có câu hỏng nào (`n_degraded = 0`) **và** trong 50 câu đối chứng, số câu fp32 đúng mà bf16 sai **≤ 2** (tức ≤ 4%).
2. Nếu không thoả, chọn **fp32** (an toàn, chậm hơn).
3. fp16 bị loại trong mọi trường hợp. Nếu trên L4 nó tái hiện **≠ 23** câu hỏng, ghi nhận như một phát hiện: tràn số phụ thuộc phần cứng.

Ngoài ra cell in **tỉ lệ trùng khớp dự đoán** giữa bf16 và fp32 trên cả 73 câu, cho biết đổi dtype làm câu trả lời thay đổi đến đâu.

*Kỳ vọng: bảng 3 dòng, một dòng `KẾT LUẬN`, biến `DTYPE` được gán cho các phần sau.*

In [ ]:
import pandas as pd
from src.eval.metrics import normalize_vi

MAX_CONTROL_REGRESS = 2      # luật định trước, không sửa sau khi xem kết quả

df = pd.DataFrame([{k: v for k, v in r.items() if k != 'degraded_ids'} for r in RESULTS])
display(df)

get  = lambda tag: next((r for r in RESULTS if f'_probe_{tag}_' in r['run_name'] and r['max_pixels'] == 401408), None)
fp16, fp32, bf16 = get('fp16'), get('fp32'), get('bf16')

regress, gain, agree = None, None, None
if fp32 and bf16 and fp32['status'] == bf16['status'] == 'ok':
    em32, em16b = em_by_id(fp32['run_name']), em_by_id(bf16['run_name'])
    regress = [q for q in CONTROL_IDS if em32[q] == 1 and em16b[q] == 0]
    gain    = [q for q in CONTROL_IDS if em32[q] == 0 and em16b[q] == 1]
    p32 = {str(r['question_id']): normalize_vi(r['prediction']) for r in load_preds(fp32['run_name'])}
    pbf = {str(r['question_id']): normalize_vi(r['prediction']) for r in load_preds(bf16['run_name'])}
    agree = sum(p32[q] == pbf[q] for q in PROBE_IDS) / len(PROBE_IDS)
    print(f'Đối chứng 50 câu — bf16 thoái lui so với fp32: {len(regress)} | cải thiện: {len(gain)}')
    print(f'Dự đoán bf16 trùng fp32 (sau chuẩn hoá): {agree:.1%} trên 73 câu')

if bf16 and bf16['status'] == 'ok' and bf16['n_degraded'] == 0 and regress is not None and len(regress) <= MAX_CONTROL_REGRESS:
    DTYPE, verdict = 'bfloat16', 'CHỌN bf16: không tràn số, không thoái lui đáng kể so với fp32.'
elif fp32 and fp32['status'] == 'ok':
    DTYPE, verdict = 'float32', 'CHỌN fp32: bf16 không đạt luật định trước.'
else:
    DTYPE, verdict = None, 'CHƯA ĐỦ DỮ LIỆU — xem lại log các nhánh.'

fp16_note = None
if fp16 and fp16['status'] == 'ok':
    fp16_note = (f"fp16 trên L4 tái hiện {fp16['n_degraded']}/23 câu hỏng"
                 + (' (khớp T4)' if fp16['n_degraded'] == 23 else ' -> KHÁC T4: tràn số phụ thuộc phần cứng, ghi vào R4'))
if fp32 and bf16 and fp32['status'] == bf16['status'] == 'ok':
    print(f"Tốc độ: fp32 {fp32['sec_per_sample']} s/câu | bf16 {bf16['sec_per_sample']} s/câu "
          f"-> bf16 nhanh hơn {fp32['sec_per_sample'] / max(bf16['sec_per_sample'], 1e-9):.2f}×")

print('\n' + '=' * 70)
print('KẾT LUẬN :', verdict)
if fp16_note:
    print('GHI NHẬN :', fp16_note)
print('DTYPE    =', DTYPE)
print('=' * 70)
DTYPE_TAG = {'bfloat16': 'bf16', 'float32': 'fp32'}.get(DTYPE)

## 8. Chi phí suy luận ở 1.280 token (dùng dtype vừa chọn)

**Vấn đề:** W6 đo trên tập dev 300 câu (T4, fp32): 1.280 token hơn 512 token **+0,039 ANLS** (có ý nghĩa thống kê), **+0,027 EM** (không có ý nghĩa, McNemar p = 0,134), nhưng **chậm 1,46×**. Phần độ chính xác đã có. Cái còn thiếu là **chi phí trên L4**. Chạy lại đúng 73 câu probe ở 1.280 token rồi so tốc độ với nhánh 512 cùng dtype.

*Kỳ vọng: 1 dòng kết quả và tỉ lệ chậm hơn của 1.280 so với 512 trên L4.*

In [ ]:
assert DTYPE, 'Chưa có DTYPE -> xem lại phần 7'
r1280 = run_variant(f'W07_probe_{DTYPE_TAG}_mp1280_qwen25vl_vitextvqa73_L4_seed42',
                    dtype=DTYPE, max_pixels=1003520)
r512 = get(DTYPE_TAG)
if r1280['status'] == 'ok':
    INFER_RATIO = r1280['sec_per_sample'] / max(r512['sec_per_sample'], 1e-9)
    print(f"\nSuy luận L4: 512 tok {r512['sec_per_sample']} s/câu | 1.280 tok {r1280['sec_per_sample']} s/câu "
          f"-> chậm {INFER_RATIO:.2f}× (T4/fp32 ở W6: 1,46×)")
else:
    INFER_RATIO = None

## 9. Smoke train QLoRA 30 step: 512 vs 1.280 token trên L4

**Vấn đề:** số epoch và `max_pixels` phụ thuộc vào **giây/step** khi huấn luyện. Trên T4/fp32 con số này là 78,8 s/step, tức khoảng 14,8 giờ/epoch ViVQA, không khả thi. Cell chạy lại đúng smoke run của W6 (200 mẫu ViVQA, 30 step, batch 2 × tích luỹ 8, gradient checkpointing bật), chỉ đổi GPU và dtype.

Adapter của smoke run **không dùng để báo cáo**. Nó chỉ dùng để đo chi phí.

*Kỳ vọng: mỗi run in tiến độ loss và kết thúc bằng `train_log.json`. Thời gian mỗi run chưa đo trên L4; trên T4 là ~40 phút, L4 kỳ vọng nhanh hơn đáng kể.*

In [ ]:
SMOKE = {
    512:  f'W07_smoke_qlora_qwen25vl_vivqa_mp512_{DTYPE_TAG}_L4_seed42',
    1280: f'W07_smoke_qlora_qwen25vl_vivqa_mp1280_{DTYPE_TAG}_L4_seed42',
}
for mp_tok, run in SMOKE.items():
    mp = mp_tok * 784
    print(f'\n▶ {run}  (max_pixels={mp})')
    !python -u -m src.adaptation.train_qlora --dataset vivqa --run_name {run} --limit 200 --max_steps 30 --max_pixels {mp} --dtype {DTYPE}

## 10. Tổng hợp + ước tính thời gian 1 epoch

Gom mọi con số vào **một bảng** và **một file** `w07_config_freeze_summary.json` để commit làm bằng chứng.

Ước tính giờ/epoch = ⌈số mẫu train / batch hiệu dụng⌉ × giây/step. Đây là **ước tính** từ 30 step (có tính cả warm-up), và đo trên ảnh ViVQA.

⚠️ **Lưu ý (phát hiện khi chạy):** ảnh ViVQA (COCO) gần như đều **nhỏ hơn** mức 512 token, nên hai smoke train ở phần 9 thực chất giống nhau, và dòng ViTextVQA trong bảng giờ/epoch là **ngoại suy sai**. Chi phí thật của ViTextVQA được đo ở phần 11.

**Cell này không tự chốt `max_pixels`.** Quyết định đó cân cả độ chính xác (W6), chi phí (L4) và ngân sách compute unit, nên sẽ chốt cùng Claude khi xem bảng.

*Kỳ vọng: bảng probe, bảng smoke train, bảng giờ/epoch cho 2 dataset × 2 độ phân giải, và đường dẫn file tóm tắt.*

In [ ]:
import math

display(pd.DataFrame([{k: v for k, v in r.items() if k != 'degraded_ids'} for r in RESULTS]))

N_TRAIN = {}
for ds, rel in [('vivqa', 'data/vivqa/train.json'), ('vitextvqa_official', 'data/vitextvqa_official/train.json')]:
    p = f'{DRIVE_ROOT}/{rel}'
    N_TRAIN[ds] = len(json.load(open(p, encoding='utf-8'))) if os.path.exists(p) else None
print('Số mẫu train:', N_TRAIN)

smoke_rows, epoch_rows = [], []
for mp_tok, run in SMOKE.items():
    path = f'{EXP_ROOT}/{run}/train_log.json'
    if not os.path.exists(path):
        smoke_rows.append({'max_tokens': mp_tok, 'status': 'no_log (OOM / bị ngắt?)'})
        continue
    m = json.load(open(path, encoding='utf-8'))['meta']
    eff = m['training']['per_device_batch_size'] * m['training']['gradient_accumulation_steps']
    sps = m['cost']['sec_per_step']
    smoke_rows.append({'max_tokens': mp_tok, 'status': 'ok', 'dtype': m['torch_dtype'],
                       'sec_per_step': sps, 'sec_per_sample': round(sps / eff, 3),
                       'peak_vram_gb': m['cost']['peak_vram_gb'], 'final_loss': m['cost']['final_loss'],
                       'gpu': m['hardware']['gpu']})
    for ds, n in N_TRAIN.items():
        if n:
            epoch_rows.append({'dataset': ds, 'max_tokens': mp_tok, 'n_train': n,
                               'steps_per_epoch': math.ceil(n / eff),
                               'hours_per_epoch': round(math.ceil(n / eff) * sps / 3600, 2)})

print('\nSmoke train (T4/fp32 ở W6: 78,8 s/step, VRAM 5,93 GB ở 512 token):')
display(pd.DataFrame(smoke_rows))
print('Ước tính giờ / epoch trên L4:')
display(pd.DataFrame(epoch_rows))

summary = {
    'purpose': 'Chốt cấu hình trên L4 trước khi chạy B1 (baseline chính thức) và B2 (QLoRA)',
    'hardware': HW,
    'probe_file': os.path.basename(PROBE_PATH), 'probe_sha256': actual_sha,
    'dtype_rule': f'bf16 nếu n_degraded=0 và thoái lui đối chứng <= {MAX_CONTROL_REGRESS}/50; ngược lại fp32',
    'dtype_verdict': verdict, 'dtype_chosen': DTYPE, 'fp16_note': fp16_note,
    'bf16_vs_fp32': {'control_regress_ids': regress, 'control_gain_ids': gain, 'prediction_agreement': agree},
    'infer_ratio_1280_vs_512': INFER_RATIO,
    'probe_runs': RESULTS,
    'smoke_train': smoke_rows, 'epoch_estimates': epoch_rows, 'n_train': N_TRAIN,
    'config_evidence': 'qwen25vl3b_config_evidence.json',
    'max_pixels_decision': 'CHƯA CHỐT — quyết định cùng Claude sau khi xem bảng',
}
with open(f'{OUT_DIR}/w07_config_freeze_summary.json', 'w', encoding='utf-8') as fp:
    json.dump(summary, fp, ensure_ascii=False, indent=2)
print('\nGhi ->', f'{OUT_DIR}/w07_config_freeze_summary.json')

## 11. Chi phí train thật trên ViTextVQA: 512 vs 1.280 token

**Vấn đề:** phần 9 đo trên ViVQA, mà ảnh ViVQA vốn nhỏ hơn ngưỡng 512 token, nên đổi `max_pixels` không làm ảnh thay đổi. Chi phí của độ phân giải cao chỉ lộ ra trên ảnh scene-text cỡ lớn của **ViTextVQA**.

Cell này làm hai việc:
1. Đo kích thước 500 ảnh train ngẫu nhiên (seed 42) mỗi dataset: bao nhiêu % ảnh bị thu nhỏ ở mỗi mức.
2. Smoke train 30 step trên **ViTextVQA** ở 512 và 1.280 token (cùng 200 mẫu, cùng batch).

**Luật định trước:** chọn 1.280 nếu train chậm **≤ 1,5×** so với 512; ngược lại giữ 512, và 1.280 thành một ablation. Căn cứ: 1.280 cho ANLS +0,039 có ý nghĩa (W6), suy luận chỉ chậm 1,16× (phần 8).

*Kỳ vọng: bảng kích thước ảnh, 2 smoke train (~10–20 phút mỗi run), một dòng `KẾT LUẬN max_pixels`.*

In [ ]:
# 11. Bổ sung: kích thước ảnh + chi phí train 512 vs 1.280 token trên ViTextVQA
import random
from PIL import Image
from src.data.vivqa_dataset import load_vivqa

MAX_TRAIN_RATIO = 1.5   # luật định trước: chọn 1.280 nếu train ViTextVQA chậm không quá 1,5× so với 512

def pixel_stats(dataset, rel, n=500):
    '''Đo số pixel của n ảnh train ngẫu nhiên (seed 42): bao nhiêu % bị max_pixels thu nhỏ.'''
    root = run_baseline.prepare_images(dataset, DRIVE_ROOT)
    recs = load_vivqa(f'{DRIVE_ROOT}/{rel}', image_dir=root)
    px = []
    for r in random.Random(42).sample(recs, min(n, len(recs))):
        with Image.open(r['image_path']) as im:
            px.append(im.size[0] * im.size[1])
    px.sort()
    return {'dataset': dataset, 'n': len(px), 'median_px': px[len(px) // 2],
            'pct_over_512tok': round(100 * sum(p > 401408 for p in px) / len(px), 1),
            'pct_over_1280tok': round(100 * sum(p > 1003520 for p in px) / len(px), 1)}

PX = [pixel_stats('vivqa', 'data/vivqa/train.json'),
      pixel_stats('vitextvqa_official', 'data/vitextvqa_official/train.json')]
display(pd.DataFrame(PX))

SMOKE_VT = {512:  'W07_smoke_qlora_qwen25vl_vitextvqa_mp512_bf16_L4_seed42',
            1280: 'W07_smoke_qlora_qwen25vl_vitextvqa_mp1280_bf16_L4_seed42'}
for mp_tok, run in SMOKE_VT.items():
    mp = mp_tok * 784
    print(f'\n▶ {run}  (max_pixels={mp})')
    !python -u -m src.adaptation.train_qlora --dataset vitextvqa_official --run_name {run} --limit 200 --max_steps 30 --max_pixels {mp} --dtype {DTYPE}

rows_vt = {}
for mp_tok, run in SMOKE_VT.items():
    path = f'{EXP_ROOT}/{run}/train_log.json'
    if os.path.exists(path):
        m = json.load(open(path, encoding='utf-8'))['meta']
        rows_vt[mp_tok] = {'max_tokens': mp_tok, 'sec_per_step': m['cost']['sec_per_step'],
                           'peak_vram_gb': m['cost']['peak_vram_gb'], 'final_loss': m['cost']['final_loss'],
                           'hours_per_epoch': round(math.ceil(35159 / 16) * m['cost']['sec_per_step'] / 3600, 2)}
display(pd.DataFrame(rows_vt.values()))

TRAIN_RATIO, mp_verdict = None, 'CHƯA ĐỦ DỮ LIỆU (thiếu train_log — OOM hoặc bị ngắt?)'
if 512 in rows_vt and 1280 in rows_vt:
    TRAIN_RATIO = rows_vt[1280]['sec_per_step'] / rows_vt[512]['sec_per_step']
    mp_verdict = (f'CHỌN 1.280 token: train ViTextVQA chậm {TRAIN_RATIO:.2f}× ≤ {MAX_TRAIN_RATIO}×'
                  if TRAIN_RATIO <= MAX_TRAIN_RATIO else
                  f'CHỌN 512 token: train ViTextVQA chậm {TRAIN_RATIO:.2f}× > {MAX_TRAIN_RATIO}× (1.280 để làm ablation)')
print('\n' + '=' * 70 + f'\nKẾT LUẬN max_pixels: {mp_verdict}\n' + '=' * 70)

with open(f'{OUT_DIR}/w07_vitextvqa_cost_summary.json', 'w', encoding='utf-8') as fp:
    json.dump({'hardware': HW, 'dtype': DTYPE, 'pixel_stats_train_n500_seed42': PX,
               'smoke_train_vitextvqa': list(rows_vt.values()),
               'rule': f'chọn 1280 nếu sec_per_step(1280)/sec_per_step(512) <= {MAX_TRAIN_RATIO}',
               'train_ratio_1280_vs_512': TRAIN_RATIO, 'verdict': mp_verdict,
               'infer_ratio_1280_vs_512_probe73': INFER_RATIO}, fp, ensure_ascii=False, indent=2)
print('Ghi ->', f'{OUT_DIR}/w07_vitextvqa_cost_summary.json')

## 12. Kiểm thử chạy tiếp (resume) trên GPU thật

**Vấn đề:** một epoch ViTextVQA dài ~11 giờ trên L4, mà Colab có thể ngắt bất cứ lúc nào. Từ W7, `train_qlora.py` lưu checkpoint định kỳ và **tự chạy tiếp** khi chạy lại đúng lệnh cũ. Kiểm thử offline đã qua (mô hình nhỏ trên CPU), nhưng phải xác nhận trên đúng model 4-bit + LoRA thật.

Cách thử: train 12 step, lưu mỗi 5 step, **giả lập Colab đứt ở step 7** (cờ `--debug_crash_at_step`). Lúc đó chỉ có `checkpoint-5`. Chạy lại đúng lệnh (bỏ cờ giả lập) → phải tiếp từ step 5 và kết thúc ở step 12.

Cell tự đủ: tự `git pull` code mới và không cần biến của các phần trước, nên chạy được cả khi runtime vừa khởi động lại (khi đó chạy phần 1 trước).

*Kỳ vọng: lần 1 báo lỗi `[debug] giả lập ngắt tại step 7` (đây là chủ đích); lần 2 in `↻ CHẠY TIẾP từ checkpoint-5`; cuối cùng `✅ RESUME HOẠT ĐỘNG`. Khoảng 10 phút.*

In [ ]:
import json, os
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline

EXP_ROOT = '/content/drive/MyDrive/ViVQA-VLM/experiments'
RUN_RS   = 'W07_resume_test_qlora_qwen25vl_vivqa_bf16_L4_seed42'   # dtype bf16 lấy từ config đã chốt
ARGS     = f'--dataset vivqa --run_name {RUN_RS} --limit 200 --max_steps 12 --save_steps 5'

print('▶ Lần 1: giả lập ngắt ở step 7 (lỗi bên dưới là CHỦ ĐÍCH)')
!python -u -m src.adaptation.train_qlora {ARGS} --debug_crash_at_step 7 2>&1 | tail -n 4
print(sorted(os.listdir(f'{EXP_ROOT}/{RUN_RS}/checkpoints')))

print('\n▶ Lần 2: chạy lại đúng lệnh -> phải chạy tiếp')
!python -u -m src.adaptation.train_qlora {ARGS}

m = json.load(open(f'{EXP_ROOT}/{RUN_RS}/train_log.json', encoding='utf-8'))['meta']
print('\nresumed_from :', m['resumed_from'])
print('global_step  :', m['cost']['global_step'], '| step phiên này:', m['cost']['steps'])
ok = m['resumed_from'] == 'checkpoint-5' and m['cost']['global_step'] == 12 and m['cost']['steps'] == 7
print('✅ RESUME HOẠT ĐỘNG' if ok else '❌ RESUME CHƯA ĐÚNG -> gửi log cho Claude, CHƯA chạy B2')

## 13. Kết luận cấu hình (đã chốt W07) và bước tiếp theo

| Quyết định | Chốt | Căn cứ (đều đo trên L4) |
|---|---|---|
| dtype | **bf16** | 0 câu hỏng; thoái lui 1/50 so với fp32 (luật ≤ 2); nhanh hơn fp32 1,71×. fp16 hỏng 23/23 như T4 |
| `max_pixels` | **401.408 (512 token)** | Train ViTextVQA ở 1.280 token chậm 1,91× (luật ≤ 1,5×); 1.280 thành ablation |
| Lượng tử hoá | **4-bit NF4** cho cả B1 và B2 | B2 là QLoRA nên B1 phải cùng điều kiện; 4-bit vs 16-bit để ablation W8+ |
| Lưu / chạy tiếp | checkpoint mỗi 200 step + adapter mỗi epoch | 1 epoch ViTextVQA ~11 giờ |

Cấu hình đã ghi vào `configs/qwen_lora.yaml` và khoá trong `CANONICAL` của `src/eval/run_baseline.py`. Mọi kết quả nằm ở `experiments/W07_*` trên Drive.

**Bước tiếp:** chạy **B1** — baseline chính thức, một lần, cả hai dataset, với cấu hình trên.